# 04 — Baseline: Custom CNN (From Scratch)

**Goal of this notebook**: train a simple CNN with no pretrained weights, on the clean
patient-level split. This is the baseline we compare transfer learning against — expect
this to underperform, since ~3k images (fewer after grouping by patient) is small for a
from-scratch CNN to learn good visual features from.

In [ ]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

import pandas as pd

from src.data_utils import build_tf_dataset
from src.evaluate import evaluate_predictions
from src.models import build_custom_cnn
from src.train import train_model

CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

## Load the pre-split metadata and build tf.data pipelines

**Intent**: reuse `build_tf_dataset` from `src/data_utils.py` — the same loader that's
covered by `tests/test_data_utils.py::test_build_tf_dataset_end_to_end` — so this
notebook and the test suite stay in sync rather than duplicating loading logic.

In [ ]:
split_metadata = pd.read_csv("../data/processed/metadata_split.csv")

train_dataset = build_tf_dataset(
    split_metadata, "train", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True
)
val_dataset = build_tf_dataset(
    split_metadata, "val", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
)
test_dataset = build_tf_dataset(
    split_metadata, "test", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

## Train

**Intent**: train for up to 30 epochs; `train_model` handles early stopping,
checkpointing the best val_accuracy, and CSV logging internally.

In [ ]:
model = build_custom_cnn(input_shape=(*IMG_SIZE, 1), num_classes=len(CLASS_NAMES))

history = train_model(
    model,
    train_dataset,
    val_dataset,
    run_name="custom_cnn_baseline",
    epochs=30,
    checkpoint_dir=Path("../models/saved_models"),
)

## Evaluate on the held-out test set

**Intent**: report per-class precision/recall/F1 and the confusion matrix — not just
overall accuracy, since confusing 'no tumor' with a tumor class is a more serious error
than confusing two tumor subtypes.

In [ ]:
import numpy as np

y_true_idx, y_pred_idx = [], []
for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)
    y_pred_idx.extend(preds.argmax(axis=1))
    y_true_idx.extend(labels.numpy())

y_true = np.array([CLASS_NAMES[i] for i in y_true_idx])
y_pred = np.array([CLASS_NAMES[i] for i in y_pred_idx])

results = evaluate_predictions(y_true, y_pred, CLASS_NAMES)
print(results["report"])
print("\nConfusion matrix:")
print(results["confusion_matrix"])
print(f"\nno_tumor_miss_rate: {results['no_tumor_miss_rate']}")

Next notebook: **05_transfer_learning_densenet.ipynb** — same data pipeline and
evaluation approach, DenseNet121 transfer learning model. Compare its test-set results
against this baseline.